# Q-learning

In this demonstration, we'll go through a complete implementation of [Q-learning](https://en.wikipedia.org/wiki/Q-learning). This is a state value based method that maintains a table of values for state-action pairs.

This table is optimized by an algorithm close to the classical [temporal difference learning](https://en.wikipedia.org/wiki/Temporal_difference_learning), with the major difference that we optimize $Q(s, a)$ and not $V(s)$ (the expected return after having performed the action $a$ in the state $s$ compared to the expected return in state $s$).

## Imports

In [ ]:
from io import StringIO
from itertools import product
from random import choice as random_choice, random
from typing import Generic, Hashable, Sequence, Tuple, TypeVar

from numpy import float32, linspace, zeros

## Types

Those types are completely optional: just here to document the expected arguments in the functions in the rest of the demonstration.

In [ ]:
StateT = TypeVar("StateT", bound=Hashable)
ActionT = TypeVar("ActionT", bound=Hashable)


class BaseEnvironment(Generic[StateT, ActionT]):
  def step(self, action: ActionT) -> Tuple[StateT, float, bool]:
    raise NotImplementedError()

  def reset(self) -> StateT:
    raise NotImplementedError()

## Q-learning

3 important functions:

- `bellman_update` implements the temporal difference update
- `episode` implements the sequence of Bellman updates of an episode
- `learn` iterates for a given number of episodes to learn the Q-table.

In [ ]:
class QLearner(Generic[StateT, ActionT]):
  def __init__(
    self,
    actions: Sequence[ActionT],
    states: Sequence[StateT],
    environment: BaseEnvironment[StateT, ActionT],
    gamma: float = 0.9,
    alpha: float = 0.1,
    n_iter: int = 100,
  ):
    self.actions = actions
    self.action_to_index = {a: i for i, a in enumerate(actions)}
    self.states = states
    self.state_to_index = {s: i for i, s in enumerate(states)}
    self.environment = environment
    self.gamma = gamma
    self.alpha = alpha
    self.n_iter = n_iter
    self.q_table = zeros((len(states), len(actions)), dtype=float32)

  def __str__(self) -> str:
    with StringIO() as string_io:
      actions_str = " | ".join(f"{str(a):^5}" for a in self.actions)
      string_io.write(f"{' ' * 10} | {actions_str}\n")
      for state, row in zip(self.states, self.q_table):
        state_values = " | ".join(f"{v:5.2f}" for v in row)
        string_io.write(f"{str(state):10} | {state_values}\n")
      return string_io.getvalue()

  def learn(self) -> float:
    len_i = len(str(self.n_iter))
    for i, epsilon in enumerate(linspace(1, 0, self.n_iter), start=1):
      self.epsilon = epsilon
      reward = self.episode()
      if i % 10 == 0:
        print(f"iteration {i:{len_i}}, ε = {epsilon:.2f}, r = {reward:7.2f}")
    return reward

  def get_q(self, state: StateT, action: ActionT) -> float:
    return self.q_table[self.state_to_index[state], self.action_to_index[action]]

  def set_q(self, state: StateT, action: ActionT, value: float) -> None:
    self.q_table[self.state_to_index[state], self.action_to_index[action]] = value

  def bellman_update(self, s: StateT, a: ActionT, r: float, new_s: StateT) -> None:
    q = self.get_q(s, a)
    best_new_s_a = self.best_action(new_s)
    new_s_max_q = self.get_q(new_s, best_new_s_a)
    self.set_q(s, a, q + self.alpha * (r + self.gamma * new_s_max_q - q))

  def best_action(self, s: StateT) -> ActionT:
    _, a = max((self.get_q(s, a), a) for a in self.actions)
    return a

  def episode(self) -> float:
    s = self.environment.reset()
    total_r = 0.0
    done = False
    while not done:
      if random() >= self.epsilon:
        # Exploitation
        a = self.best_action(s)
      else:
        # Exploration
        a = random_choice(self.actions)
      new_s, r, done = self.environment.step(a)
      total_r += r
      self.bellman_update(s, a, r, new_s)
      s = new_s
    return total_r

## Usage

The main thing we have to define here is the step function of our environment. It should return the next state when action $a$ is performed, the obtained reward and whether the task is done or not.

In [ ]:
class Environment(BaseEnvironment):
  def __init__(self) -> None:
    self.allowed = frozenset(
      {
        (0, 1),
        (0, 3),
        (0, 4),
        (0, 5),
        (1, 1),
        (1, 3),
        (2, 1),
        (2, 2),
        (2, 3),
        (2, 4),
        (2, 5),
        (2, 6),
        (3, 1),
        (4, 1),
      }
    )

  def reset(self) -> Tuple[int, int]:
    self.current_state = (2, 0)
    return self.current_state

  def step(self, action: str) -> Tuple[Tuple[int, int], float, bool]:
    r, c = self.current_state
    if action == "↑":
      r -= 1
    elif action == "→":
      c += 1
    elif action == "↓":
      r += 1
    elif action == "←":
      c -= 1
    else:
      raise ValueError("Invalid action")
    if 0 <= r < 5 and 1 <= c < 6 and (r, c) in self.allowed:
      self.current_state = (r, c)
      return (r, c), -1, False
    elif (r, c) == (2, 6):
      self.current_state = (r, c)
      return (r, c), -1, True
    else:
      return self.current_state, -1, False


def test_qlearn() -> None:
  actions = ["↑", "→", "↓", "←"]
  states = [(i, j) for i, j in product(range(5), range(1, 6))] + [(2, 0), (2, 6)]

  environment = Environment()

  q_learner: QLearner[Tuple[int, int], str] = QLearner(
    actions=actions, states=states, environment=environment, n_iter=1000
  )
  final_reward = q_learner.learn()
  assert int(final_reward) == -6
  print(q_learner)

In [ ]:
test_qlearn()